# 考古模型：直觉版路线

## 一句话

你要做的不是“从零训练一个新大模型”。

你要做的是：

```text
一个会按考古方法工作的模型
+
一个能随时查考古资料的知识库
```

更准确地说：

```text
考古助手 = RAG 资料库 + 微调后的模型 + 证据规则 + 聊天界面
```

用户感觉像“模型懂考古”。

实际后台是：

```text
模型先查资料，再按考古规则回答。
```

---

## 1. RAG 是什么

RAG 可以理解成：

```text
给模型装一个可搜索的考古图书馆。
```

模型本身不需要背下所有书。

它回答问题前，系统自动去资料库里找相关内容。

比如你问：

```text
M12 这座墓可能是什么年代？
```

后台会自动找：

```text
M12 的原始描述
M12 的出土器物
M12 的地层关系
相关墓葬分期方法
相关器物类型学资料
术语表和证据规则
```

然后再让模型回答。

所以 RAG 的本质是：

```text
知识不塞进模型脑子里，而是放在资料库里，让模型随用随查。
```

---

## 2. 微调是什么

微调不是让模型背书。

微调是教模型：

```text
应该怎么做事。
```

比如教它：

```text
如何抽取墓葬信息
如何写遗址摘要
如何区分事实、推断、假设
如何列证据
如何避免乱断代
如何按你的格式输出
```

微调后的模型更像一个受过训练的研究助理。

它不一定知道所有资料。

但它知道：

```text
看到资料后应该怎么分析。
```

---

## 3. 训练模型是什么

训练模型通常指从零造一个大模型。

这需要：

```text
海量文本
大量 GPU
分布式训练
模型架构
tokenizer
安全对齐
评测体系
部署系统
很多钱和时间
```

一个人通常不做这个。

对你来说，训练模型不是第一选择。

你的第一选择应该是：

```text
RAG + 微调
```

---

## 4. 三者的关系

可以这样理解：

```text
RAG = 给模型资料
微调 = 教模型方法
训练 = 从零造模型
```

对考古项目来说：

```text
知识放 RAG
方法放微调
规则写 prompt
```

这是最实用的组合。

---

## 5. 考古资料应该怎么分

你的资料大致分成几类。

### 事实资料

这些放进 RAG：

```text
发掘报告
考古简报
论文
墓葬表格
器物登记表
图版说明
地层记录
测年数据
遗址平面图说明
铭文释文
```

它们回答：

```text
有什么？在哪里？出土了什么？报告怎么说？
```

### 方法资料

这些也可以放进 RAG，也可以做成微调样本：

```text
类型学方法
地层学原则
断代方法
器物分类规则
墓葬分期流程
考古报告写作规范
空间分析方法
```

它们回答：

```text
应该怎么判断？怎么分析？怎么写？
```

### 术语表

例如：

```text
灰坑 / H
墓葬 / M
探方 / T
打破关系
叠压关系
随葬品组合
鬲、豆、罐、鼎、盉
```

作用：

```text
减少误读，统一词汇。
```

### 证据规则

这是最重要的。

例如：

```text
不能凭感觉断代
必须引用来源
不确定就标“待核”
地层关系优先于器物风格
冲突资料必须并列
区分原文事实和模型推断
```

---

## 6. 最推荐的资料结构

```text
archaeology_model/
  corpus/
    facts/
      excavation_reports/
      site_reports/
      tomb_tables/
      artifact_catalogs/
      captions/
      inscriptions/
    methods/
      typology/
      stratigraphy/
      dating/
      spatial_analysis/
      report_writing/
    glossary/
      terms.md
      artifact_names.md
    workflows/
      tomb_analysis.md
      artifact_classification.md
      site_summary.md
    rules/
      evidence_policy.md
      citation_policy.md
```

---

## 7. 元数据为什么重要

RAG 不是简单地把文本丢进去。

每段资料最好带元数据。

例如：

```json
{
  "title": "XX遗址发掘报告",
  "source_type": "excavation_report",
  "site": "XX遗址",
  "region": "河南",
  "period": "战国",
  "page": 123,
  "feature_id": "M12",
  "topic": ["墓葬", "陶器", "分期"]
}
```

没有元数据，模型很难知道：

```text
这段话来自哪里
说的是哪个遗址
对应哪一页
是不是权威报告
和哪个墓葬有关
```

考古研究里，出处非常重要。

所以元数据不是装饰。

它是证据链的一部分。

---

## 8. 实际工作流

整体流程：

```text
PDF / Word / 图片
↓
OCR 或 PDF 解析
↓
清洗文本
↓
按墓葬、器物、图版、表格行切片
↓
加元数据
↓
建向量索引
↓
用户提问
↓
系统检索相关资料
↓
模型按规则回答
```

---

## 9. 切片怎么切

不要只按字数切。

考古资料最好按研究对象切。

例如：

```text
一个墓葬一段
一个灰坑一段
一个器物描述一段
一个地层单位一段
一个图版说明一段
一个表格行一段
```

因为你的问题通常是：

```text
M12 是什么？
这个器物属于哪类？
某遗址有哪些墓葬？
这组墓葬怎么分期？
```

对象级切片比随机切片更好。

---

## 10. 微调样本长什么样

微调样本不是整本书。

微调样本是：

```text
输入 → 理想输出
```

例如：

```json
{
  "instruction": "从考古报告段落中抽取墓葬信息。",
  "input": "M12，竖穴土坑墓，墓向180°，出土陶鬲1、陶豆2、陶罐1。",
  "output": {
    "遗迹编号": "M12",
    "墓葬形制": "竖穴土坑墓",
    "墓向": "180°",
    "随葬品": ["陶鬲", "陶豆", "陶罐"],
    "年代判断": "资料不足，不能单独断代",
    "证据": "原文仅提供形制、墓向和随葬品，缺少地层关系、图版比较或测年数据。"
  }
}
```

再比如：

```json
{
  "instruction": "判断下面考古论述中的事实、推断和假设。",
  "input": "M8 出土陶鬲、豆、罐组合，可能属战国早期。",
  "output": {
    "事实": ["M8 出土陶鬲、豆、罐组合"],
    "推断": ["可能属战国早期"],
    "假设": [],
    "不确定性": "未提供地层关系、图版比较或报告原始年代意见。"
  }
}
```

---

## 11. 微调适合教什么

适合：

```text
墓葬字段抽取
器物名称规范化
遗迹类型分类
事实/推断/假设分类
固定格式摘要
证据化回答格式
考古报告风格改写
```

不适合：

```text
记住几千篇报告
记住每一页内容
替代资料库
保证所有断代正确
```

---

## 12. 一个自然的最终体验

用户只看到：

```text
请分析 M12 的年代。
```

系统后台做：

```text
1. 查 M12 原文
2. 查同遗址相关墓葬
3. 查相关器物类型学资料
4. 查断代方法
5. 查证据规则
6. 让微调模型回答
```

用户看到的答案像这样：

```text
M12 可暂按战国早期处理，但证据不足。

依据：
1. 原报告记录 M12 为竖穴土坑墓。
2. 随葬品组合包括陶鬲、陶豆、陶罐。
3. 该组合与本区战国墓葬常见组合接近。

不确定性：
1. 未见明确地层关系。
2. 未提供可复核图版比较。
3. 无测年数据。

结论等级：初步判断，需复核。
```

这就是你想要的“自然”。

不是用户手动调用 RAG。

而是系统默认自动检索。

---

## 13. 常用模型选择

一个人做中文考古，优先：

```text
Qwen2.5-7B-Instruct
Qwen2.5-14B-Instruct
Qwen3-8B
Qwen3-14B
```

机器一般：

```text
Qwen2.5-7B-Instruct + QLoRA
```

显存好一点：

```text
Qwen2.5-14B-Instruct + QLoRA
```

不建议一开始做：

```text
70B 微调
从零训练
继续预训练
```

---

## 14. 常用工具

### RAG

```text
LlamaIndex
LangChain
RAGFlow
Dify
AnythingLLM
Open WebUI Documents
```

### 向量库

```text
Milvus Lite：本地零配置，未来可无缝升级
Chroma：本地最省事（备选）
Qdrant：长期项目更稳
pgvector：已有 PostgreSQL 时用
Milvus Distributed：大型系统用
```

### 文档解析

```text
MinerU
PyMuPDF
PaddleOCR
Tesseract
marker-pdf
docling
```

### 微调

```text
LLaMA-Factory
Unsloth
Axolotl
Transformers + PEFT
```

### 部署

```text
Ollama
llama.cpp
vLLM
Open WebUI
Dify
```

---

## 15. 最小个人方案

先别做大系统。

先做这个：

```text
50-100 篇考古 PDF
↓
解析成文本
↓
做 RAG
↓
写 500 条高质量微调样本
↓
用 Qwen2.5-7B-Instruct 做 QLoRA
↓
把微调模型和 RAG 接到一个聊天界面
```

---

## 16. 推荐组合

最小可行：

```text
模型：Qwen2.5-7B-Instruct
RAG：LlamaIndex
向量库：Milvus Lite / Chroma（备选）
解析：MinerU / PyMuPDF
微调：LLaMA-Factory
部署：Ollama
界面：Open WebUI
```

更稳版本：

```text
模型：Qwen2.5-14B-Instruct
RAG：LlamaIndex / RAGFlow
向量库：Qdrant
微调：LLaMA-Factory / Unsloth
部署：vLLM
界面：Dify / 自己写 Web UI
```

---

## 17. 最重要的判断

如果你的目标是：

```text
让模型知道大量考古资料
```

用 RAG。

如果你的目标是：

```text
让模型按你的考古流程工作
```

用微调。

如果你的目标是：

```text
造一个全新的考古大模型
```

通常不要做。

---

## 18. 最终原则

```text
知识不要硬塞进模型参数。
知识放资料库。
方法用微调教。
规则写进 prompt。
引用靠 RAG 保证。
```

这条路线最适合一个人做。